# CatRanger demo
Detect a cat, keep its identity, and say how far it is. Runs on the provided Go2 inference set (`how_far`/`mental_map`) or any video/webcam.

Run from the repo root. Install: `make install-ml` (or `pip install -e .[ml]`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import cv2, numpy as np
import matplotlib.pyplot as plt
from catranger.config import load_app
from catranger.pipeline import CatRanger
from catranger.viz import draw
from catranger.io import frame_source

## 1. Configure
Loads the Go2 intrinsics and the cat+distance task config. Approach A = YOLO11, B = RT-DETR.

In [ ]:
app = load_app('cat_distance')
print('camera:', app.camera.name, '| fx', app.camera.fx, '| classes', app.classes)
ranger = CatRanger(app, approach='approach_a', use_depth=True)

## 2. Run on the provided inference set
Point `SOURCE` at the contest `how_far` stills (or a cat video). We overlay the per-cat distance ± confidence interval.

In [ ]:
SOURCE = os.path.expanduser('~/Downloads/inference_sets_contest/how_far')  # or 'data/cat_demo.mp4'
rows = []
shown = 0
for idx, frame in frame_source(SOURCE, max_frames=6):
    result = ranger.process(frame, frame_index=idx)
    annotated = draw(getattr(ranger, 'last_undistorted', frame), result)
    for o in result.observations:
        rows.append({'frame': idx, 'id': o.track_id,
                     'dist_m': round(o.distance.meters, 2) if o.distance else None,
                     'bearing_deg': round(o.bearing_deg, 1)})
    if shown < 3:
        plt.figure(figsize=(11, 6)); plt.axis('off')
        plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)); plt.title(f'frame {idx}'); plt.show()
        shown += 1
rows

## 3. Compare the two approaches (performance report)
The deck requires ≥ 2 approaches compared. This is the table the judges want.

In [ ]:
import time
from catranger.eval.metrics import fps_stats
summary = {}
for ap in ['approach_a', 'approach_b']:
    r = CatRanger(app, approach=ap, use_depth=True)
    times = []
    for idx, frame in frame_source(SOURCE, max_frames=6):
        t0 = time.perf_counter(); r.process(frame, idx); times.append(time.perf_counter() - t0)
    summary[ap] = fps_stats(times)
summary